# PydanticAI: Production-Ready Agents

In previous lessons, we implemented agents with bare OpenAI API calls and ToyAIKit. ToyAIKit is great for prototyping, learning, and debugging. For production, I recommend using a dedicated agentic framework.

Pydantic AI is my favorite agentic framework. Most agentic frameworks are similar, so if you learn one, moving to another isn't difficult.

You can find the documentation at [ai.pydantic.dev](https://ai.pydantic.dev)


## Setup (from previous lessons)

Rebuild the index, highlighter, file index, `SearchTools` instance, and instructions from the previous lesson so this notebook runs on its own.


In [1]:
import nest_asyncio

# Windows + Jupyter: top-level await with pydantic-ai can hard-crash the kernel.
# nest_asyncio + run_sync is the stable workaround (lesson uses await elsewhere).
nest_asyncio.apply()

from dotenv import load_dotenv

load_dotenv()

from gitsource import GithubRepositoryDataReader
from minsearch import AppendableIndex, Highlighter, Tokenizer
from minsearch.tokenizer import DEFAULT_ENGLISH_STOP_WORDS
from search_tools import SearchTools

In [ ]:
reader = GithubRepositoryDataReader(
    repo_owner="evidentlyai",
    repo_name="docs",
    allowed_extensions={"md", "mdx"},
)
files = reader.read()
parsed_docs = [doc.parse() for doc in files]

index = AppendableIndex(
    text_fields=["title", "description", "content"],
    keyword_fields=["filename"],
)
index.fit(parsed_docs)

stopwords = DEFAULT_ENGLISH_STOP_WORDS | {"evidently"}
highlighter = Highlighter(
    highlight_fields=["content"],
    max_matches=3,
    snippet_size=50,
    tokenizer=Tokenizer(stemmer="snowball", stop_words=stopwords),
)

file_index = {doc["filename"]: doc["content"] for doc in parsed_docs}

search_tools = SearchTools(index, highlighter, file_index)

In [ ]:
instructions = """
You're a documentation assistant.

Answer the user question using only the documentation knowledge base.

Make 3 iterations:

1) First iteration:
   - Perform one search using the search tool to identify potentially relevant documents.
   - Explain (in 2-3 sentences) why this search query is appropriate for the user question.

2) Second iteration:
   - Analyze the results from the previous search.
   - Based on the filenames or documents returned, perform:
       - Up to 2 additional search queries to refine or expand coverage, and
       - One or more get_file calls to retrieve the full content of the most relevant documents.
   - For each search or get_file call, explain (in 2-3 sentences) why this action is necessary and how it helps answer the question.

3) Third iteration:
   - Analyze the retrieved document contents from get_file.
   - Synthesize the information from these documents into a final answer to the user.

IMPORTANT:
- At every step, explicitly explain your reasoning for each search query or file retrieval.
- Use only facts found in the documentation knowledge base.
- Do not introduce outside knowledge or assumptions.
- If the answer cannot be found in the retrieved documents, clearly inform the user.

Additional notes:
- The knowledge base is entirely about Evidently, so you do not need to include the word "evidently" in search queries.
- Prefer retrieving and analyzing full documents (via get_file) before producing the final answer.
"""

## Creating the Agent

For tools, we can either list them manually:


In [ ]:
tools = [search_tools.search, search_tools.get_file]

Or use a helper function. The `get_instance_methods` function from ToyAIKit extracts all methods from a class automatically:


In [ ]:
from toyaikit.tools import get_instance_methods

tools = get_instance_methods(search_tools)

We briefly looked at this code in the previous lesson.

We get the tools automatically. I typically use this approach because it gives me flexibility to add another tool to the tool class and the agent automatically picks it up.

Now import the Agent class:


In [ ]:
from pydantic_ai import Agent

Create the agent:


In [ ]:
search_agent = Agent(
    name='search',
    model='openai:gpt-4o-mini',
    instructions=instructions,
    tools=tools
)

The agent needs:
- **name** - identifier for the agent
- **model** - which LLM to use (OpenAI, Anthropic, Gemini, Grok, etc.)
- **instructions** - how to accomplish the task
- **tools** - what the agent can use


## Running the Agent

Set up our query:


In [ ]:
query = 'how do I use evidently to monitor my machine learning models?'

Run the agent. Pydantic AI is asynchronous — the lesson uses `await search_agent.run(...)`.

On Windows Jupyter/Cursor, that `await` can kill the kernel, so we use `run_sync` with `nest_asyncio` instead (same result):


In [ ]:
result = search_agent.run_sync(query)

Get the output:


In [ ]:
print(result.output)

The agent runs the same agentic search pattern we defined in the instructions.


## Inspecting Messages

Let's look at what the agent returns. We can access all messages:


In [ ]:
messages = result.all_messages()

And the usage information:


In [ ]:
result.usage

Let's inspect the message structure:


In [ ]:
for m in messages:
    print(m.kind)
    for p in m.parts:
        part_kind = p.part_kind
        if part_kind == 'user-prompt':
            print('USER:', p.content)
        if part_kind == 'tool-call':
            print('TOOL CALL:', p.tool_name, p.args)
        if part_kind == 'tool-return':
            print('TOOL RETURN:', p.tool_name)
        if part_kind == 'text':
            print(p.content)
    print()

Each message has a kind (request or response) and parts. The parts can be:
- **user-prompt** - what we sent
- **tool-call** - the model's decision to invoke a tool
- **tool-return** - the result we sent back
- **text** - the model's response

This interface is clear. You can see exactly what was sent to OpenAI, what came back, and trace through the full conversation.


## Message History

For a follow-up question, we pass the previous messages:


In [ ]:
new_prompt = 'show me the code'

Run with message history:


In [ ]:
result2 = search_agent.run_sync(new_prompt, message_history=messages)

Get the new output:


In [ ]:
print(result2.output)

We can also inspect just the new messages. First, define a helper function:


In [ ]:
def print_messages(messages):
    for m in messages:
        print(m.kind)
        for p in m.parts:
            part_kind = p.part_kind
            if part_kind == 'user-prompt':
                print('USER:', p.content)
            if part_kind == 'tool-call':
                print('TOOL CALL:', p.tool_name, p.args)
            if part_kind == 'tool-return':
                print('TOOL RETURN:', p.tool_name)
            if part_kind == 'text':
                print(p.content)
        print()

In [ ]:
print_messages(result2.new_messages())

## Interactive QA Loop

Let's wrap this in a simple QA loop. Reuse the `print_messages` helper from above.

Set up the loop:


In [ ]:
from pydantic_ai.usage import RunUsage

messages = []
usage = RunUsage()

while True:
    user_prompt = input('You:')
    if user_prompt.lower().strip() == 'stop':
        break

    result = search_agent.run_sync(user_prompt, message_history=messages)
    usage = usage + result.usage

    print_messages(result.new_messages())
    messages.extend(result.new_messages())

This keeps track of total usage across the entire conversation and shows progress as it goes.

Show the final usage:


In [ ]:
usage

## Event Stream Handlers

When running an agent with `agent.run()`, we wait without knowing what's happening. The agent makes tool calls, but we have no visibility into the process.

Pydantic AI provides an `event_stream_handler` parameter to see what's happening during execution. Let's create a callback handler.

Define the `NamedCallback` class:


In [ ]:
from pydantic_ai.messages import FunctionToolCallEvent


class NamedCallback:

    def __init__(self, agent):
        self.agent_name = agent.name

    async def print_function_calls(self, ctx, event):
        # Detect nested streams
        if hasattr(event, "__aiter__"):
            async for sub in event:
                await self.print_function_calls(ctx, sub)
            return

        if isinstance(event, FunctionToolCallEvent):
            tool_name = event.part.tool_name
            args = event.part.args
            print(f"TOOL CALL ({self.agent_name}): {tool_name}({args})")

    async def __call__(self, ctx, event):
        return await self.print_function_calls(ctx, event)

This class is callable (has `__call__`) and handles nested event streams. When it detects a `FunctionToolCallEvent`, it prints the agent name, tool name, and arguments.

Create the callback instance:


In [ ]:
callback = NamedCallback(search_agent)

Pass it to the agent:

(Lesson uses `await search_agent.run(...)`; on Windows Jupyter we keep `run_sync` so the kernel doesn't die.)


In [ ]:
result = search_agent.run_sync(
    "how do I build dashboads with evidently?",
    event_stream_handler=callback
)

When you run this, you'll see the tool calls as they happen:

```text
TOOL CALL (search): search({"query":"build dashboards"})
TOOL CALL (search): search({"query": "dashboard management"})
TOOL CALL (search): get_file({"filename": "docs/platform/dashboard_add_panels.mdx"})
```

One thing to note: when you use `event_stream_handler`, the request becomes streaming internally. This is just how Pydantic AI is implemented.

This pattern gives you visibility into what your agent is doing in real-time, which is especially useful for debugging and monitoring.


## Summary

We migrated from ToyAIKit (prototyping) to Pydantic AI (production-ready):
- Agent setup with model, instructions, and tools
- `get_instance_methods` helper for automatic tool discovery
- Async/await pattern for running agents
- Clear message structure for debugging
- Message history for multi-turn conversations
- Usage tracking across conversations
- `event_stream_handler` for live visibility into tool calls

The concepts are the same across frameworks. You define the brain (model), how to accomplish tasks (instructions), tools, and memory (message history). Pydantic AI just provides a production-ready interface.
